# Laboratório: harness de IA em 60 minutos

Objetivo: completar um controle e testar permissão de ferramenta/caminho, aprovação humana e trace. O fluxo usa **replay preparado**: as ações propostas são fixas, sem chamada a modelo; controles e validadores executam de fato.

Entradas docentes: CSV do exercício aprovado da Fase 2, replay.json e referência aprovada. Saídas: resumo.json somente após aprovação e trace.json. Biblioteca padrão do Python, CPU. No Colab, carregue os três arquivos quando o seletor abrir; não inclua a referência no pacote dos participantes. Não use terminal, editor local ou Drive pessoal.

Solução docente: aprovar_acao confere decisão e identidade da ação concreta. Todos os testes devem passar.


## 1. Preparar ambiente e conferir entradas

Objetivo: identificar Python, exercício, replay e referência docente por hash. No Colab, o seletor recebe exatamente os três arquivos; a preparação guarda a referência em docente/dados/referencia/ da sessão.


In [1]:
import csv, hashlib, json, math, platform, statistics
from datetime import datetime, timezone
from pathlib import Path
RAIZ=Path.cwd().resolve()
DADO=Path("dados/chuva_rs_2018_2025_exercicio.csv")
REPLAY=Path("replay.json")
REFERENCIA_REL=Path("docente/dados/referencia/chuva_rs_2018_2025_referencia.csv")
REFERENCIA=(RAIZ.parent/REFERENCIA_REL if RAIZ.name=="participantes" else RAIZ/REFERENCIA_REL)
HASH_DADO="f948fb1c00c9f684d9b080ba0cdad35b653ebe5ea1268d64c778f4026b6b153f"
HASH_REPLAY="fef76b95637350f65f470d0bfaffeb2cd53981490372c92269fbf3633575704a"
HASH_REFERENCIA="aaf01e630f8b86a9303c8ce3e2799c6c1e29e7056eec5c9aa01767c7454fa4c3"
def sha256(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
if not (RAIZ/DADO).is_file() or not (RAIZ/REPLAY).is_file() or not REFERENCIA.is_file():
    try: from google.colab import files
    except ImportError as erro:
        raise FileNotFoundError("Abra em participantes com a referência docente adjacente ou carregue três arquivos no Colab.") from erro
    entrada=files.upload()
    destinos={DADO.name:RAIZ/DADO, REPLAY.name:RAIZ/REPLAY, REFERENCIA_REL.name:REFERENCIA}
    if set(entrada)!=set(destinos): raise ValueError("Carregue exatamente exercício, replay.json e referência docente.")
    for nome,destino in destinos.items():
        destino.parent.mkdir(parents=True,exist_ok=True)
        destino.write_bytes(entrada[nome])
assert sha256(RAIZ/DADO)==HASH_DADO, "CSV não aprovado"
assert sha256(RAIZ/REPLAY)==HASH_REPLAY, "replay incorreto"
assert sha256(REFERENCIA)==HASH_REFERENCIA, "referência docente incorreta"
print("Python",platform.python_version(),"| raiz",RAIZ)
print("Dados",HASH_DADO,"| replay",HASH_REPLAY,"| referência docente",HASH_REFERENCIA)


Python 3.13.7 | raiz /Users/ricardoceneviva/Library/CloudStorage/GoogleDrive-ceneviva@gmail.com/Meu Drive/Data-Science/06_cursos_e_aulas/2026-10-07_harness_ia/participantes
Dados f948fb1c00c9f684d9b080ba0cdad35b653ebe5ea1268d64c778f4026b6b153f | replay fef76b95637350f65f470d0bfaffeb2cd53981490372c92269fbf3633575704a


## 2. Ferramentas, argumentos e caminhos

O registro aceita somente leitura e gravação predefinidas. Os argumentos devem ter as chaves e os tipos esperados. A permissão é aplicada em código após resolver o caminho, inclusive travessia de diretórios. Não há execução de código arbitrário.


In [2]:
PACOTE=json.loads((RAIZ/REPLAY).read_text(encoding="utf-8"))
assert PACOTE["tipo"]=="replay_preparado"
ACOES=PACOTE["acoes"]
assert isinstance(ACOES,list) and len(ACOES)==2
ESTADO={}; TRACE=[]
def ler_dados(path):
    with (RAIZ/path).open(newline="",encoding="utf-8") as f:
        leitor=csv.DictReader(f); linhas=list(leitor); colunas=leitor.fieldnames
    ESTADO["linhas"]=linhas
    return {"linhas":len(linhas),"colunas":colunas,"sha256":sha256(RAIZ/path)}
def gravar_saida(path,conteudo):
    destino=RAIZ/path
    destino.parent.mkdir(parents=True,exist_ok=True)
    destino.write_text(conteudo,encoding="utf-8")
    return {"arquivo":path,"bytes":len(conteudo.encode("utf-8"))}
TOOLS={"ler_dados":(ler_dados,{"path"}),"gravar_saida":(gravar_saida,{"path","conteudo"})}
BASES={"ler_dados":"dados","gravar_saida":"saida"}
def selecionar(acao):
    if not isinstance(acao,dict) or set(acao)!={"tool","args"}:
        raise ValueError("ação inválida")
    nome,args=acao["tool"],acao["args"]
    if not isinstance(nome,str) or nome not in TOOLS or not isinstance(args,dict):
        raise ValueError("ferramenta não permitida")
    ferramenta,campos=TOOLS[nome]
    if set(args)!=campos or not isinstance(args["path"],str):
        raise TypeError("argumentos ou path inválidos")
    if nome=="gravar_saida":
        if not isinstance(args["conteudo"],str): raise TypeError("conteudo deve ser texto")
        try: resumo=json.loads(args["conteudo"])
        except json.JSONDecodeError as erro: raise ValueError("JSON inválido") from erro
        if (not isinstance(resumo,dict) or set(resumo)!={"media_precipitacao_mm"}
            or type(resumo["media_precipitacao_mm"]) not in (int,float)
            or not math.isfinite(resumo["media_precipitacao_mm"])):
            raise ValueError("resumo inválido")
        if "linhas" not in ESTADO: raise ValueError("leia dados antes de gravar")
    return ferramenta
def validar_permissao(acao):
    caminho=Path(acao["args"]["path"])
    if caminho.is_absolute(): raise PermissionError("caminho absoluto bloqueado")
    base=(RAIZ/BASES[acao["tool"]]).resolve()
    destino=(RAIZ/caminho).resolve()
    if destino==base or not destino.is_relative_to(base):
        raise PermissionError("caminho bloqueado")
    return str(destino.relative_to(RAIZ))


## 3. Validadores internos

Após a leitura, o harness confere schema, chaves, meses, ausências, valores e cobertura. Após a gravação, recalcula a média sob a regra declarada: primeira linha por região-mês, descartando chuva vazia. Esses checks são automáticos. A média não é uma estimativa causal nem uma correção dos valores ausentes.


In [3]:
COLUNAS=["codigo_rgi","nome_rgi","mes","precipitacao_mm","cobertura_area"]
MESES={f"{ano}-{mes:02d}" for ano in range(2018,2026) for mes in range(1,13)}
def checar_dados(linhas,colunas):
    if colunas!=COLUNAS: raise ValueError("schema inesperado")
    chaves=[(r["codigo_rgi"],r["mes"]) for r in linhas]
    regioes={r["codigo_rgi"] for r in linhas}
    if any(len(x)!=6 or not x.isdigit() for x in regioes):
        raise ValueError("código inválido")
    if any(r["mes"] not in MESES for r in linhas):
        raise ValueError("mês fora do período")
    for r in linhas:
        if r["precipitacao_mm"] and not math.isfinite(float(r["precipitacao_mm"])):
            raise ValueError("chuva não finita")
        if not 0<=float(r["cobertura_area"])<=1: raise ValueError("cobertura inválida")
    return {"linhas":len(linhas),"regioes":len(regioes),
            "meses":len({r["mes"] for r in linhas}),
            "duplicadas":len(chaves)-len(set(chaves)),
            "ausentes":sum(r["precipitacao_mm"]=="" for r in linhas),
            "negativos":sum(bool(r["precipitacao_mm"]) and float(r["precipitacao_mm"])<0 for r in linhas),
            "chaves_faltantes":len(regioes)*len(MESES)-len(set(chaves)),
            "cobertura_incompleta":sum(float(r["cobertura_area"])<1 for r in linhas)}
def media_apos_regra(linhas):
    vistos=set(); valores=[]
    for r in linhas:
        chave=(r["codigo_rgi"],r["mes"])
        if chave in vistos: continue
        vistos.add(chave)
        if r["precipitacao_mm"] and float(r["precipitacao_mm"])>=0:
            valores.append(float(r["precipitacao_mm"]))
    return {"linhas_usadas":len(valores),"media_recalculada":math.fsum(valores)/len(valores)}
def conferir_resultado(linhas,conteudo):
    calculo=media_apos_regra(linhas)
    reportada=json.loads(conteudo)["media_precipitacao_mm"]
    erro=abs(reportada-calculo["media_recalculada"])
    return {**calculo,"media_reportada":reportada,
            "erro_absoluto":erro,"confere":erro<1e-6}


## 4. Complete um controle: aprovação da ação concreta

A autorização vem **fora do replay** e inclui decisão e SHA-256 da ação completa, com ferramenta, caminho e conteúdo. A função inicial recusa tudo de forma segura. Troque apenas o retorno para aceitar um dicionário cuja decisão seja aprovar e cujo acao_sha256 coincida com identidade_acao(acao).


In [4]:
def identidade_acao(acao):
    serial=json.dumps(acao,ensure_ascii=False,sort_keys=True,separators=(",",":"))
    return hashlib.sha256(serial.encode("utf-8")).hexdigest()
def aprovar_acao(acao,autorizacao):
    # TODO DIDÁTICO ÚNICO: conferir decisão e hash da ação exata.
    return (isinstance(autorizacao,dict) and autorizacao.get("decisao")=="aprovar"
            and autorizacao.get("acao_sha256")==identidade_acao(acao))


## 5. Fluxo único e trace

A sequência é: validar proposta, aplicar permissão, exigir aprovação antes da gravação, executar ferramenta, validar automaticamente e registrar. Toda tentativa gera evento estruturado. Uma recusa registra a tentativa, mas não chama a ferramenta.


In [5]:
def executar(acao,autorizacao=None):
    evento={"momento_utc":datetime.now(timezone.utc).isoformat(),
            "agente":"replay_preparado","acao":acao,
            "acao_sha256":identidade_acao(acao),
            "permission":"not_checked","approval":"not_required",
            "status":"blocked","result":None,"checks":None,"reason":None}
    try:
        ferramenta=selecionar(acao)
        evento["path_resolvido"]=validar_permissao(acao)
        evento["permission"]="allowed"
        if acao["tool"]=="gravar_saida":
            evento["approval"]="approved" if aprovar_acao(acao,autorizacao) else "denied"
            if evento["approval"]=="denied":
                evento["reason"]="aprovação recusada para esta ação"
                TRACE.append(evento); return evento
        resultado=ferramenta(**acao["args"])
        evento["result"]=resultado
        evento["checks"]=(checar_dados(ESTADO["linhas"],resultado["colunas"])
                         if acao["tool"]=="ler_dados" else
                         conferir_resultado(ESTADO["linhas"],acao["args"]["conteudo"]))
        evento["status"]="executed"
    except (ValueError,TypeError,PermissionError,FileNotFoundError) as erro:
        if isinstance(erro,PermissionError): evento["permission"]="denied"
        evento["reason"]=f"{type(erro).__name__}: {erro}"
    TRACE.append(evento)
    return evento
def simular_decisao_humana(acao,aprovar):
    # Fixture automática identificada: decisão externa ao replay.
    print("Decisão simulada:",acao["tool"],acao["args"].get("path"),
          "aprovar" if aprovar else "recusar")
    return {"decisao":"aprovar" if aprovar else "recusar",
            "acao_sha256":identidade_acao(acao)}
def solicitar_aprovacao_humana(acao):
    # Uso opcional em aula; não é chamado por Executar tudo.
    print("Ação concreta para revisão:")
    print(json.dumps(acao,ensure_ascii=False,indent=2))
    resposta=input("Aprovar exatamente esta ação? [s/N] ").strip().lower()
    return {"decisao":"aprovar" if resposta=="s" else "recusar",
            "acao_sha256":identidade_acao(acao)}


## 6. Testar os três controles e os validadores

A bateria executa a leitura permitida, ferramenta proibida, caminhos fora da allowlist e com traversal, argumento inválido, recusa, aprovação, autorização vinculada, checks de dados e comparação numérica. A autorização positiva é simulada no teste automático. No uso real, uma pessoa deve inspecionar a ação concreta antes de autorizar.


In [6]:
TESTES={}
leitura=executar(ACOES[0])
esperado={"linhas":4134,"regioes":43,"meses":96,"duplicadas":6,"ausentes":41,
          "negativos":0,"chaves_faltantes":0,"cobertura_incompleta":0}
TESTES["ação permitida"]=leitura["status"]=="executed" and leitura["result"]["sha256"]==HASH_DADO
TESTES["duplicatas e ausências"]=leitura["checks"]==esperado
proibidas=[
 {"tool":"executar_shell","args":{"cmd":"echo proibido"}},
 {"tool":"ler_dados","args":{"path":"../docente/VALIDACAO.md"}},
 {"tool":"ler_dados","args":{"path":"dados/../../docente/VALIDACAO.md"}},
 {"tool":"gravar_saida","args":{"path":"dados/chuva_rs_2018_2025_exercicio.csv","conteudo":'{"media_precipitacao_mm": 0.0}'}},
 {"tool":"ler_dados","args":{"path":42}}]
hash_antes=sha256(RAIZ/DADO)
bloqueios=[executar(x) for x in proibidas]
TESTES["ferramenta proibida"]=bloqueios[0]["status"]=="blocked"
TESTES["caminho e traversal"]=all(x["status"]=="blocked" and x["permission"]=="denied" for x in bloqueios[1:4])
TESTES["argumentos inválidos"]=bloqueios[4]["status"]=="blocked"
TESTES["exercício preservado"]=sha256(RAIZ/DADO)==hash_antes
print(json.dumps(TESTES,ensure_ascii=False,indent=2))


{
  "ação permitida": true,
  "duplicatas e ausências": true,
  "ferramenta proibida": true,
  "caminho e traversal": true,
  "argumentos inválidos": true,
  "exercício preservado": true
}


### Recusa e aprovação

A recusa mira arquivo inexistente; seu caminho deve continuar ausente. O teste positivo usa a ação do replay. Uma segunda tentativa altera o conteúdo com a mesma autorização: deve ser bloqueada. Na versão inicial, só a aprovação positiva está pendente.


In [7]:
recusado=RAIZ/"saida/recusa.json"
if recusado.exists(): raise RuntimeError("Saída antiga de recusa: remova antes do teste.")
acao_recusa={"tool":"gravar_saida","args":{"path":"saida/recusa.json",
             "conteudo":ACOES[1]["args"]["conteudo"]}}
negativa=executar(acao_recusa,simular_decisao_humana(acao_recusa,False))
TESTES["recusa sem efeito"]=negativa["status"]=="blocked" and negativa["approval"]=="denied" and not recusado.exists()
autorizacao=simular_decisao_humana(ACOES[1],True)
positiva=executar(ACOES[1],autorizacao)
TESTES["aprovação concreta"]=positiva["status"]=="executed" and positiva["approval"]=="approved" and (RAIZ/"saida/resumo.json").is_file()
alterada=json.loads(json.dumps(ACOES[1]))
alterada["args"]["conteudo"]='{"media_precipitacao_mm": 0.0}'
reuso=executar(alterada,autorizacao)
TESTES["autorização não reutilizável"]=reuso["status"]=="blocked" and reuso["approval"]=="denied"
print({k:TESTES[k] for k in ("recusa sem efeito","aprovação concreta","autorização não reutilizável")})


Decisão simulada: gravar_saida saida/recusa.json recusar
Decisão simulada: gravar_saida saida/resumo.json aprovar
{'recusa sem efeito': True, 'aprovação concreta': True, 'autorização não reutilizável': True}


### Resultado calculado e comparado

Uma segunda implementação com statistics.fmean confere a média do harness. O replay contém a média reportada; o validador confronta ambas. Um valor zero deve falhar na comparação. Na solução, o mesmo check aparece no evento da gravação.


In [8]:
vistos=set(); valores=[]
for r in ESTADO["linhas"]:
    chave=(r["codigo_rgi"],r["mes"])
    if chave not in vistos:
        vistos.add(chave)
        if r["precipitacao_mm"]: valores.append(float(r["precipitacao_mm"]))
independente=statistics.fmean(valores)
certo=conferir_resultado(ESTADO["linhas"],ACOES[1]["args"]["conteudo"])
errado=conferir_resultado(ESTADO["linhas"],'{"media_precipitacao_mm": 0.0}')
TESTES["cálculo e comparação"]=(len(valores)==4087
    and abs(independente-certo["media_recalculada"])<1e-10
    and certo["confere"] is True and errado["confere"] is False)
TESTES["validators internos"]=(leitura["checks"]==esperado and
    (not TESTES["aprovação concreta"] or positiva["checks"]==certo))
print("Linhas:",len(valores),"Média (mm):",independente,
      "Erro do replay:",certo["erro_absoluto"])


Linhas: 4087 Média (mm): 151.7622034117935 Erro do replay: 0.0


### Conferir e guardar o trace

Todo evento registra ação, argumentos, identidade, permissão, aprovação, estado, resultado, checks ou motivo. O JSON salvo permite auditoria. A única pendência admitida no esqueleto é a aprovação positiva; se outra falhar, interrompa e examine o evento correspondente.


In [9]:
campos={"momento_utc","agente","acao","acao_sha256","permission",
        "approval","status","result","checks","reason"}
TESTES["trace completo"]=(len(TRACE)==1+len(proibidas)+3
 and all(campos<=set(e) and e["acao_sha256"]==identidade_acao(e["acao"]) for e in TRACE)
 and all((e["checks"] is not None and e["result"] is not None)
         if e["status"]=="executed" else bool(e["reason"]) for e in TRACE))
(RAIZ/"saida").mkdir(exist_ok=True)
(RAIZ/"saida/trace.json").write_text(json.dumps({
 "tipo":"trace_operacional","replay_sha256":HASH_REPLAY,"dados_sha256":HASH_DADO,
 "python":platform.python_version(),"eventos":TRACE},
 ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
print(json.dumps(TESTES,ensure_ascii=False,indent=2))
pendentes=[k for k,v in TESTES.items() if not v]
print("Pendências:",pendentes or "nenhuma")
assert set(pendentes)<={"aprovação concreta"}, "Falha inesperada"


{
  "ação permitida": true,
  "duplicatas e ausências": true,
  "ferramenta proibida": true,
  "caminho e traversal": true,
  "argumentos inválidos": true,
  "exercício preservado": true,
  "recusa sem efeito": true,
  "aprovação concreta": true,
  "autorização não reutilizável": true,
  "cálculo e comparação": true,
  "validators internos": true,
  "trace completo": true
}
Pendências: nenhuma


## 7. Interpretar e reexecutar

Complete a função da seção 4 e use Executar tudo. Todos os testes devem ser true. Abra saida/trace.json e localize a leitura, os bloqueios, a recusa e a gravação. O hash prova qual ação foi aprovada. A limpeza deixa 4.087 linhas, mas não recupera as 41 chuvas ausentes; a referência aprovada permanece no domínio docente.

Reinicie o ambiente antes de repetir. O teste desta fase é local; uma sessão limpa do Colab e a repetição integrada do replay pertencem à Fase 5.


## Conferência docente com a referência aprovada

Compara todas as chaves e valores presentes, sem imputação ou deduplicação da referência. Esta célula permanece no domínio docente.


In [10]:
assert sha256(REFERENCIA)==HASH_REFERENCIA
with REFERENCIA.open(newline="",encoding="utf-8") as f: referencia=list(csv.DictReader(f))
mapa={(r["codigo_rgi"],r["mes"]):r for r in referencia}
assert len(mapa)==4128
assert {(r["codigo_rgi"],r["mes"]) for r in ESTADO["linhas"]}==set(mapa)
divergentes=sum(r["precipitacao_mm"]!="" and r!=mapa[(r["codigo_rgi"],r["mes"])] for r in ESTADO["linhas"])
media_referencia=statistics.fmean(float(r["precipitacao_mm"]) for r in referencia)
assert divergentes==0 and sum(r["precipitacao_mm"]=="" for r in ESTADO["linhas"])==41
assert abs(media_referencia-151.643325125969)<1e-9
assert all(TESTES.values())
print("Referência:",len(mapa),"chaves; valores presentes divergentes:",divergentes)
print("Média da referência:",media_referencia,"mm")
print("Diferença do exercício filtrado:",independente-media_referencia,"mm")


Referência: 4128 chaves; valores presentes divergentes: 0
Média da referência: 151.643325125969 mm
Diferença do exercício filtrado: 0.11887828582450766 mm
